In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [5]:
%%writefile src/stability.py
import numpy as np
import pandas as pd

from src import config, model, rules, splits
from src.metrics import pythag

def team_strengths(teams):
    t=teams.set_index('team')
    a,b=config.TS_TEAMS
    return t.loc[a,'pythag'] , t.loc[b,'pythag']


Writing src/stability.py


In [11]:
%%writefile -a src/stability.py
# ---------- 實驗 1：模擬次數與亂數 ----------
def exp_seeds(teams, n_seeds=30, n=10_000):
    pa, pb = team_strengths(teams)
    probs = [model.simulate_series(pa, pb, config.TS_HOME_PATTERN, config.TS_NEED,
                                   config.HOME_ADVANTAGE, n=n, seed=s)[0] for s in range(n_seeds)]
    return pd.Series(probs, name="奪冠機率")

Appending to src/stability.py


In [13]:

import pandas as pd
from src import stability as st
s = st.exp_seeds(pd.read_csv('data/clean/teams.csv'))
print(f'平均 {s.mean():.1%}、標準差 {s.std():.2%}、最小 {s.min():.1%}、最大 {s.max():.1%}')


平均 67.1%、標準差 0.46%、最小 65.8%、最大 67.8%


In [16]:
%%writefile -a src/stability.py
# ---------- 實驗 2：參數敏感度 ----------
def exp_sensitivity(teams):
    """一次只改一個參數，看系列賽奪冠機率和中立場地單場勝率怎麼變。"""
    t = teams.set_index("team")
    a, b = config.TS_TEAMS
    rows = []
    settings = [("主場優勢", v, dict(hfa=v)) for v in [0, 0.015, 0.03, 0.045, 0.06]] + \
               [("畢氏指數", v, dict(exp=v)) for v in [1.8, 1.83, 1.9, 2.0]]
    for name, value, kw in settings:
        exp = kw.get("exp", 1.83)
        hfa = kw.get("hfa", config.HOME_ADVANTAGE)
        pa = pythag(t.loc[a, "RS"], t.loc[a, "RA"], exp)
        pb = pythag(t.loc[b, "RS"], t.loc[b, "RA"], exp)
        series = model.simulate_series(pa, pb, config.TS_HOME_PATTERN, config.TS_NEED, hfa)[0]
        rows.append({"參數": name, "數值": value, "單場勝率（中立）": model.log5(pa, pb), "奪冠機率": series})
    return pd.DataFrame(rows)

Appending to src/stability.py


In [17]:

import pandas as pd
from src import stability as st
print(st.exp_sensitivity(pd.read_csv('data/clean/teams.csv')).round(3).to_string())


     參數     數值  單場勝率（中立）   奪冠機率
0  主場優勢  0.000     0.570  0.657
1  主場優勢  0.015     0.570  0.668
2  主場優勢  0.030     0.570  0.681
3  主場優勢  0.045     0.570  0.696
4  主場優勢  0.060     0.570  0.710
5  畢氏指數  1.800     0.569  0.674
6  畢氏指數  1.830     0.570  0.676
7  畢氏指數  1.900     0.573  0.681
8  畢氏指數  2.000     0.577  0.686


In [20]:
%%writefile -a src/stability.py
def default_game_prob(teams, batters, pitchers, lg, sp_innings=6, a_home=True):
    """用「兩隊投球局數最多的先發＋打席最多的 9 人」算單場勝率，給實驗 2 測先發局數用。"""
    t = teams.set_index("team")
    a, b = config.TS_TEAMS
    ctx = model.league_context(teams, pitchers, lg["wOBA"])
    tp = pd.DataFrame(model.team_pitching(pitchers)).set_index("abbr")
    side = {}
    for team in (a, b):
        abbr = t.loc[team, "abbr"]
        sp = pitchers[(pitchers["abbr"] == abbr) & (pitchers["role"] == "先發")].nlargest(1, "IP").iloc[0]
        lu = batters[batters["abbr"] == abbr].nlargest(9, "PA")
        side[team] = (abbr, sp, lu)
    (aa, sa, la), (ab, sb, lb) = side[a], side[b]
    w = lambda lu, sp: lu.apply(lambda r: rules.bat_woba(r, sp["throws"]), axis=1).mean()
    ra = model.expected_runs(w(la, sb), la["wOBA_calc"].mean(), sb["FIP_calc"], tp.loc[ab, "bullpen_fip"],
                             tp.loc[ab, "DER"], ctx, sp_innings, a_home)
    rb = model.expected_runs(w(lb, sa), lb["wOBA_calc"].mean(), sa["FIP_calc"], tp.loc[aa, "bullpen_fip"],
                             tp.loc[aa, "DER"], ctx, sp_innings, not a_home)
    return model.win_prob_from_runs(ra, rb)

Appending to src/stability.py


In [21]:

from src import stability as st
from src.data import load
teams, batters, pitchers, const, lg = load()
for i in (5, 6, 7):
    print(i, '局：', round(st.default_game_prob(teams, batters, pitchers, lg, sp_innings=i), 3))


5 局： 0.506
6 局： 0.5
7 局： 0.494
